Загрузка датасета

In [219]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel('../data/Onlineretail.xlsx')

Обзор датасета

In [ ]:
print(df.head())
df.info()
print(df.describe())
df.isna().sum()

Переименование столбцов

In [221]:
df.columns = df.columns.str.lower()

Поиск отрицательных и равных нулю значений

In [ ]:
print(df[df['quantity'] <= 0])
print(df[df['unitprice'] < 0])
print(df[df['unitprice'] == 0])
print(df[df['unitprice'] > 30000])

Выделяющиеся коды товара и работа с ними

In [ ]:
print(df['stockcode'][df['stockcode'].str.len() > 8].unique())
print(df['stockcode'][df['stockcode'].str.len() < 5].unique())

In [ ]:
print(df[df['stockcode'] == 'POST'])
print(df[df['stockcode'] == 'D'])
print(df[df['stockcode'] == 'C2'])
print(df[df['stockcode'] == 'DOT'])
print(df[df['stockcode'] == 'M'])
print(df[df['stockcode'] == 'S'])
print(df[df['stockcode'] == 'm'])
print(df[df['stockcode'] == 'PADS'])
print(df[df['stockcode'] == 'B'])
print(df[df['stockcode'] == 'CRUK'])
print(df[df['stockcode'] == 'BANK CHARGES'].head())
print(df[df['stockcode'] == 'AMAZONFEE'].head())
print(df[df['stockcode'].str.contains('gift', na=False)].head())

Добавление необходимых для анализа столбцов и очистка от не товаров

In [225]:
df['revenue'] = df['quantity'] * df['unitprice']
df['month'] = df['invoicedate'].dt.month_name()
df2 = df.copy()
df2 = df2[~df2['stockcode'].isin(['POST', 'D', 'C2', 'DOT', 'M', 'S', 'm', 'PADS', 'B', 'CRUK',
                                  'BANK CHARGES', 'AMAZONFEE', 'gift_0001_40',
                                  'gift_0001_50', 'gift_0001_30', 'gift_0001_20'])]

Работа со столбцом 'description'

In [ ]:
descriptions = (df2.groupby('description', dropna=False).agg(
    rows = ('description', 'size'),
    total_revenue = ('revenue', 'sum'),
    min_quantity = ('quantity', 'min'),
    max_quantity = ('quantity', 'max'),
    avg_quantity = ('quantity', 'mean'),
    min_price = ('unitprice', 'min'),
    max_price = ('unitprice', 'max'),
    avg_price = ('unitprice', 'mean')
)
).sort_values('rows', ascending=False).reset_index()

print(descriptions['description'][(descriptions['rows'] < 20) & \
                                  (descriptions['description'].str.len() < 18) & \
                                    ((descriptions['total_revenue'] > 200) | \
                                     (descriptions['total_revenue'] < -200))].unique(), '\n')
print(descriptions['description'][(descriptions['rows'] < 20) & \
                                  (descriptions['description'].str.len() < 18)].unique())

Топ месяцев по продажам

In [ ]:
best_month = (df2.groupby('month').agg({'revenue': 'sum', 'quantity': 'sum'})
                                      .reset_index().sort_values('revenue', ascending=False))
print(best_month, '\n')
print(best_month['revenue'].sum())

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(best_month['month'], best_month['revenue'], marker='o', color='navy', linewidth=2)
plt.title('Monthly revenue')
plt.xlabel('Month')
plt.ylabel('Revenue, £')
plt.ticklabel_format(axis='y', style='plain')
plt.grid(True)
plt.show()

Топ 10 стран по продажам

In [ ]:
countries_revenue = df2.groupby('country')['revenue'].sum().reset_index()\
    .sort_values('revenue', ascending=False).reset_index()
countries_revenue.drop(columns=['index'], inplace=True)
print(countries_revenue.head(10))
print(countries_revenue)

In [ ]:
countries = countries_revenue['country'].head(10).copy()
revenue = countries_revenue['revenue'].head(10).copy()
countries_sorted = sorted(zip(countries, revenue), key=lambda x: x[1])
countries, revenue = zip(*countries_sorted)
plt.figure(figsize=(12, 7))
plt.barh(countries, revenue, color='teal')
plt.title('Top 10 contries revenue')
plt.xlabel('Revenue, £')
plt.ticklabel_format(axis='x', style='plain')
plt.show()

Топ 10 продуктов по сумме продаж

In [ ]:
top_prods = df2.groupby('description').agg({'revenue': 'sum', 'quantity': 'sum'}).reset_index()\
    .sort_values('revenue', ascending=False).reset_index()
top_prods.drop(columns=['index'], inplace=True)
print(top_prods.head(10))

In [ ]:
prods = top_prods['description'].head(10).copy()
sales = top_prods['revenue'].head(10).copy()
prods_sorted = sorted(zip(prods, sales), key=lambda x: x[1])
prods, sales = zip(*prods_sorted)
plt.figure(figsize=(12, 6))
plt.barh(prods, sales, color='teal')
plt.title('Top 10 prods revenue')
plt.xlabel('Revenue, £')
plt.ticklabel_format(axis='x', style='plain')
plt.show()

Распределение стоимости заказов

In [ ]:
orders = df2.groupby('invoiceno').agg({'revenue': 'sum', 'month': 'first',
                                       'quantity': 'sum'}).reset_index()\
    .sort_values('revenue', ascending=False).reset_index()
orders.drop(columns=['index'], inplace=True)

print(orders.head())
print(orders.tail(), '\n')
print(orders['revenue'].mean())
print(orders['revenue'].median(), '\n')

In [ ]:
# Проверка заказов, похожих на возвраты
print(df2[df2['invoiceno'] == 581483])
print(df2[df2['invoiceno'] == 'C581484'])

In [ ]:
positive_orders = orders[orders['revenue'] > 0]

positive_orders['revenue'][positive_orders['revenue'] <= 2000].hist(bins=50)
plt.xlabel('Order value')
plt.ylabel('Number of orders')
plt.title('Distribution of order values (<=2000)')
plt.show()

positive_orders['revenue'][positive_orders['revenue'] > 2000].hist(bins=50)
plt.xlabel('Order value')
plt.ylabel('Number of orders')
plt.title('Distribution of order values (>2000)')
plt.show()

Количество заказов по месяцам

In [ ]:
orders_month = positive_orders.groupby('month')['invoiceno'].count().reset_index()\
    .sort_values('invoiceno', ascending=False)
orders_month.rename(columns={'invoiceno': 'invoice_col'}, inplace=True)
print(orders_month, '\n')
print(orders_month['invoice_col'].sum())
print(df2['customerid'].nunique(), '— количество уникальных покупателей')

Работа с выбросами в заказах, средним отклонением и перцентилями

In [ ]:
print(positive_orders['revenue'].mean())
print(positive_orders['revenue'].median())
print(positive_orders['revenue'].std())
q1 = positive_orders['revenue'].quantile(0.25)
print(q1)
q3 = positive_orders['revenue'].quantile(0.75)
print(q3)
print(positive_orders['revenue'].quantile(0.95))
iqr = q3 - q1
print(iqr)
upper = q3 + 1.5*iqr
upper3 = q3 + 3*iqr
print(upper)
print(positive_orders[positive_orders['revenue'] > upper].tail(10), '\n')
print((positive_orders['revenue'] > upper).sum(), '\n')
print(upper3)
print(positive_orders[positive_orders['revenue'] > upper3].tail(10), '\n')
print((positive_orders['revenue'] > upper3).sum())

Нахождение корреляции между количеством товаров и суммой заказа без учета экстремальных сумм заказов (95%)

In [ ]:
revenue_in95 = positive_orders[positive_orders['revenue'] < 1509.34]
plt.figure(figsize=(12, 6))
plt.scatter(revenue_in95['revenue'], revenue_in95['quantity'], alpha=0.25, s=10)
plt.title('Correlation between revenue and quantity')
plt.xlabel('Revenue')
plt.ylabel('Quantity')
plt.show()

print(revenue_in95[['revenue', 'quantity']].corr())